In [ ]:
import sys
from pathlib import Path

NOTEBOOK_PATH_CANDIDATES = [Path.cwd(), Path.cwd() / "AgentWorkshop" / "Notebook"]
for candidate in NOTEBOOK_PATH_CANDIDATES:
    if (candidate / "workshop_bootstrap.py").exists():
        resolved_candidate = str(candidate.resolve())
        if resolved_candidate not in sys.path:
            sys.path.insert(0, resolved_candidate)

from workshop_bootstrap import build_workshop_config

CONFIG_OVERRIDES = {
    "resource_group_name": "",
    "location": "",
    "subscription_id": "",
    "foundry_account_name": "",
    "foundry_project_name": "",
    "foundry_project_endpoint": "",
    "search_service_name": "",
    "storage_account_name": "",
    "application_insights_name": "",
    "model_zone": "",
}

config = build_workshop_config({k: v for k, v in CONFIG_OVERRIDES.items() if v})
config.show()

# Workshop 2: Foundry IQ Knowledge Sources And Knowledge Base

This notebook mirrors docs/knowledge_base.md and creates Foundry IQ objects with code.

Flow:
1. Parse workshop files from data/Coffee
2. Create Search indexes
3. Create Foundry IQ knowledge sources over those indexes
4. Create a knowledge base that binds the knowledge sources and chat model
5. Output MCP endpoint for downstream agents

In [ ]:
# Uncomment this cell in a clean kernel.
# %pip install --quiet requests azure-identity pypdf

In [ ]:
import csv
import os
from pathlib import Path

from pypdf import PdfReader

from workshop_bootstrap import (
    SearchRestClient,
    WorkshopConstants,
    create_knowledge_base,
    create_knowledge_source,
    create_search_index,
    sanitize_name,
    upload_documents,
)

DATA_ROOT = Path("../../data/Coffee").resolve()
MAX_PDF_CHARACTERS = 12000
MAX_CSV_ROWS = 500
MAX_CSV_FIELDS = 20
KNOWLEDGE_BASE_NAME = "health-effects-kb"

CHAT_DEPLOYMENT_NAME = os.getenv("CHAT_DEPLOYMENT_NAME", "").strip()
CHAT_MODEL_NAME = os.getenv("CHAT_MODEL_NAME", CHAT_DEPLOYMENT_NAME).strip()
AZURE_OPENAI_ENDPOINT = (os.getenv("AZURE_OPENAI_ENDPOINT", "").strip() or config.openai_resource_endpoint).rstrip("/")

if not CHAT_DEPLOYMENT_NAME:
    raise ValueError("Set CHAT_DEPLOYMENT_NAME in your environment before running this cell.")
if not CHAT_MODEL_NAME:
    raise ValueError("Set CHAT_MODEL_NAME in your environment before running this cell.")
if not AZURE_OPENAI_ENDPOINT:
    raise ValueError("AZURE_OPENAI_ENDPOINT or AZURE_AI_PROJECT_ENDPOINT must be set.")

SOURCE_SPECS = [
    {"knowledge_source_name": "health-effects-ks", "folder": DATA_ROOT / "HealthEffects", "source_type": "health-effects"},
    {"knowledge_source_name": "coffee-recipes-ks", "folder": DATA_ROOT / "CoffeeRecipes", "source_type": "coffee-recipes"},
    {"knowledge_source_name": "coffee-csv-generalhealth-ks", "folder": DATA_ROOT / "CoffeeCSV" / "GeneralHealth", "source_type": "coffee-csv-generalhealth"},
    {"knowledge_source_name": "coffee-csv-mentalhealth-ks", "folder": DATA_ROOT / "CoffeeCSV" / "mentalHealth", "source_type": "coffee-csv-mentalhealth"},
]

def load_pdf_document(file_path: Path, source_type: str) -> dict[str, str]:
    reader = PdfReader(str(file_path))
    text_parts = [page.extract_text() or "" for page in reader.pages]
    text_value = "\n".join(text_parts).strip()
    if len(text_value) > MAX_PDF_CHARACTERS:
        text_value = text_value[:MAX_PDF_CHARACTERS]
    return {
        "id": sanitize_name(f"{source_type}-{file_path.stem}"),
        "title": file_path.name,
        "content": text_value,
        "sourcePath": str(file_path),
        "sourceType": source_type,
    }

def load_csv_documents(file_path: Path, source_type: str) -> list[dict[str, str]]:
    documents: list[dict[str, str]] = []
    with file_path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        field_names = reader.fieldnames or []
        selected_fields = field_names[:MAX_CSV_FIELDS]
        for row_index, row in enumerate(reader):
            if row_index >= MAX_CSV_ROWS:
                break
            line_parts = [f"{field}={row.get(field, '')}" for field in selected_fields]
            documents.append({
                "id": sanitize_name(f"{source_type}-{file_path.stem}-{row_index}"),
                "title": f"{file_path.name} row {row_index}",
                "content": "; ".join(line_parts),
                "sourcePath": str(file_path),
                "sourceType": source_type,
            })
    return documents

def build_documents(folder: Path, source_type: str) -> list[dict[str, str]]:
    documents: list[dict[str, str]] = []
    for file_path in sorted(folder.rglob("*")):
        if not file_path.is_file():
            continue
        suffix = file_path.suffix.lower()
        if suffix == ".pdf":
            documents.append(load_pdf_document(file_path, source_type))
            continue
        if suffix == ".csv":
            documents.extend(load_csv_documents(file_path, source_type))
            continue
        if suffix in {".md", ".txt"}:
            text_value = file_path.read_text(encoding="utf-8")
            documents.append({
                "id": sanitize_name(f"{source_type}-{file_path.stem}"),
                "title": file_path.name,
                "content": text_value,
                "sourcePath": str(file_path),
                "sourceType": source_type,
            })
    return documents

client = SearchRestClient(config.search_endpoint)
knowledge_source_names: list[str] = []

for spec in SOURCE_SPECS:
    source_name = spec["knowledge_source_name"]
    source_folder = spec["folder"]
    source_type = spec["source_type"]

    if not source_folder.exists():
        raise FileNotFoundError(f"Source folder not found: {source_folder}")

    index_name = sanitize_name(f"{source_name}-index")
    source_documents = build_documents(source_folder, source_type)
    if not source_documents:
        raise ValueError(f"No source documents generated for {source_name}")

    print(f"Creating index and knowledge source for {source_name} with {len(source_documents)} document(s).")
    create_search_index(client, index_name)
    upload_documents(client, index_name, source_documents)
    create_knowledge_source(client, source_name, index_name)
    knowledge_source_names.append(source_name)

create_knowledge_base(
    client=client,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    knowledge_source_names=knowledge_source_names,
    azure_openai_resource_uri=AZURE_OPENAI_ENDPOINT,
    chat_deployment_name=CHAT_DEPLOYMENT_NAME,
    chat_model_name=CHAT_MODEL_NAME,
)

KB_MCP_ENDPOINT = f"{config.search_endpoint}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={WorkshopConstants.SEARCH_API_VERSION}"
print("Knowledge source names:", knowledge_source_names)
print("Knowledge base name:", KNOWLEDGE_BASE_NAME)
print("Knowledge base MCP endpoint:", KB_MCP_ENDPOINT)

In [ ]:
import shutil
from workshop_bootstrap import run_command

if shutil.which("azd"):
    run_command(["azd", "env", "set", "KB_MCP_ENDPOINT", KB_MCP_ENDPOINT])
    print("Stored KB_MCP_ENDPOINT in azd environment.")
else:
    print("azd is not installed. Set KB_MCP_ENDPOINT manually if you are using toolbox-based hosted agents.")

## Notes

- This notebook creates the Foundry IQ knowledge sources and knowledge base from workshop files.
- For hosted agent toolbox wiring, continue with the agents notebooks and the Foundry quickstart commands if needed.